# 📚 Technique 55: Document Chunking

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amerob/ultimate-prompt-engineering-playbook/blob/main/notebooks/07-retrieval/55_document_chunking.ipynb)

**Category:** 07 - Retrieval-Augmented Generation
**Technique #:** 55
**Difficulty:** Intermediate

## 📋 Description

**Document Chunking** is the process of splitting large documents into smaller, semantically meaningful segments that can be efficiently stored, retrieved, and processed by LLMs. Proper chunking is critical for effective RAG systems as it directly impacts retrieval quality and answer accuracy.

### When to Use:
- When documents exceed LLM **context window limits**
- For **semantic search** requiring granular retrieval
- When you need to **preserve context** across document sections
- For **optimizing embedding quality** (smaller chunks = more focused embeddings)
- When building **knowledge bases** from long-form content

## 🔧 How It Works

```
┌─────────────────────────────────────────────────────────────────┐
│                   DOCUMENT CHUNKING STRATEGIES                  │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  STRATEGY 1: FIXED-SIZE CHUNKING                                │
│  ┌─────────────────────────────────────────────────────────┐   │
│  │ [==========Chunk 1==========][==========Chunk 2========│   │
│  │ Fixed token/character count per chunk                   │   │
│  │ ✓ Simple, predictable  ✗ May split semantic units       │   │
│  └─────────────────────────────────────────────────────────┘   │
│                                                                 │
│  STRATEGY 2: RECURSIVE CHARACTER CHUNKING                     │
│  ┌─────────────────────────────────────────────────────────┐   │
│  │ Split by: Paragraphs → Sentences → Words → Characters   │   │
│  │ Respects natural boundaries, maintains coherence        │   │
│  └─────────────────────────────────────────────────────────┘   │
│                                                                 │
│  STRATEGY 3: SEMANTIC CHUNKING                                │
│  ┌─────────────────────────────────────────────────────────┐   │
│  │ [Topic A Chunk][Topic B Chunk][Topic C Chunk]           │   │
│  │ Groups by semantic similarity, topic boundaries         │   │
│  └─────────────────────────────────────────────────────────┘   │
│                                                                 │
│  STRATEGY 4: HIERARCHICAL CHUNKING                            │
│  ┌─────────────────────────────────────────────────────────┐   │
│  │ Parent: [Section 1] → Children: [Para 1][Para 2][Para 3]│   │
│  │ Maintains parent-child relationships for context        │   │
│  └─────────────────────────────────────────────────────────┘   │
│                                                                 │
│  OVERLAP (All Strategies):                                    │
│  ┌─────────────────────────────────────────────────────────┐   │
│  │ [Chunk 1-----][-----Chunk 2-----][-----Chunk 3]         │
│  │     ↑ overlap ↑    ↑ overlap ↑                          │   │
│  │ Preserves context across chunk boundaries               │   │
│  └─────────────────────────────────────────────────────────┘   │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

### Chunk Size Guidelines:
| Use Case | Recommended Size | Overlap |
|----------|------------------|---------|
| Question Answering | 200-400 tokens | 10-20% |
| Summarization | 500-1000 tokens | 5-10% |
| Code Documentation | 300-500 tokens | 15-25% |
| Legal Documents | 400-600 tokens | 10-15% |

## ⚙️ Setup

In [ ]:
# Install required packages
!pip install -q langchain tiktoken openai

In [ ]:
import os
from getpass import getpass
from langchain.text_splitter import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    TokenTextSplitter
)
import tiktoken

# Setup API key (for potential LLM usage)
api_key = getpass("Enter your OpenAI API key: ")
os.environ["OPENAI_API_KEY"] = api_key

# Helper function to count tokens
def count_tokens(text, model="gpt-3.5-turbo"):
    encoding = tiktoken.encoding_for_model(model)
    return len(encoding.encode(text))

## 💡 Basic Example

Different chunking strategies in action.

In [ ]:
# Sample document
sample_document = """
Introduction to Machine Learning

Machine learning is a subset of artificial intelligence that enables computers to learn and improve from experience without being explicitly programmed. This revolutionary technology has transformed numerous industries.

Supervised Learning

In supervised learning, the algorithm learns from labeled training data. The model makes predictions based on the input data and is corrected when its predictions are wrong. Common applications include spam detection and image classification.

Unsupervised Learning

Unsupervised learning deals with unlabeled data. The algorithm tries to find hidden patterns and structures in the data. Clustering and dimensionality reduction are common unsupervised learning tasks.

Deep Learning

Deep learning uses neural networks with multiple layers to model complex patterns. These networks can automatically learn representations from data. Convolutional neural networks excel at image processing, while recurrent networks handle sequential data.

Applications

Machine learning powers recommendation systems, autonomous vehicles, medical diagnosis, and natural language processing. The technology continues to evolve rapidly with new architectures and training methods.
"""

print(f"Original document: {count_tokens(sample_document)} tokens\n")

# Strategy 1: Fixed-size Character Splitting
print("=== STRATEGY 1: FIXED-SIZE CHARACTER SPLITTING ===")
char_splitter = CharacterTextSplitter(
    separator="\n\n",
    chunk_size=500,
    chunk_overlap=50,
    length_function=len
)
char_chunks = char_splitter.split_text(sample_document)
print(f"Number of chunks: {len(char_chunks)}")
for i, chunk in enumerate(char_chunks[:3]):
    print(f"\nChunk {i+1} ({len(chunk)} chars):")
    print(chunk[:150] + "..." if len(chunk) > 150 else chunk)

print("\n" + "="*60 + "\n")

# Strategy 2: Recursive Character Splitting
print("=== STRATEGY 2: RECURSIVE CHARACTER SPLITTING ===")
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=30,
    separators=["\n\n", "\n", ". ", " ", ""],
    length_function=len
)
recursive_chunks = recursive_splitter.split_text(sample_document)
print(f"Number of chunks: {len(recursive_chunks)}")
for i, chunk in enumerate(recursive_chunks[:3]):
    print(f"\nChunk {i+1} ({len(chunk)} chars):")
    print(chunk[:150] + "..." if len(chunk) > 150 else chunk)

## 🌍 Real-World Example

Processing a legal document with appropriate chunking for RAG.

In [ ]:
# Sample legal document
legal_document = """
SERVICE AGREEMENT

This Service Agreement ("Agreement") is entered into as of January 1, 2024 ("Effective Date") between TechCorp Inc. ("Provider") and ClientSoft LLC ("Client").

1. DEFINITIONS

1.1 "Services" means the software development and consulting services described in Exhibit A.
1.2 "Deliverables" means all work product, documents, and materials created by Provider in performing the Services.
1.3 "Confidential Information" means any and all non-public information disclosed by either party.

2. SCOPE OF SERVICES

2.1 Provider shall perform the Services in a professional and workmanlike manner in accordance with industry standards.
2.2 Provider shall assign qualified personnel to perform the Services. Client may request replacement of any personnel with reasonable cause.
2.3 Provider shall provide weekly status reports to Client regarding progress of the Services.

3. COMPENSATION

3.1 Client shall pay Provider the fees set forth in Exhibit B.
3.2 Payment terms are Net 30 days from invoice date.
3.3 Late payments subject to 1.5% monthly service charge.

4. INTELLECTUAL PROPERTY

4.1 Client retains all rights to pre-existing materials provided to Provider.
4.2 Provider assigns all rights in Deliverables to Client upon full payment.
4.3 Provider retains rights to general methodologies and know-how.

5. CONFIDENTIALITY

5.1 Each party agrees to maintain confidentiality of the other party's Confidential Information.
5.2 Confidentiality obligations survive termination for a period of five (5) years.
5.3 Exceptions apply to publicly available information and independently developed information.

6. TERM AND TERMINATION

6.1 This Agreement commences on the Effective Date and continues for twelve (12) months.
6.2 Either party may terminate with ninety (90) days written notice.
6.3 Upon termination, Provider shall deliver all completed Deliverables and work in progress.
"""

# Legal-optimized chunking strategy
legal_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,  # Smaller chunks for precise retrieval
    chunk_overlap=80,  # Higher overlap for clause continuity (20%)
    separators=[
        "\n\n",      # Paragraph breaks (highest priority)
        "\n",        # Line breaks
        ". ",         # Sentence boundaries
        " ",          # Word boundaries
        ""            # Character fallback
    ],
    length_function=len
)

legal_chunks = legal_splitter.split_text(legal_document)

print(f"Legal Document Processing\n")
print(f"Original size: {count_tokens(legal_document)} tokens")
print(f"Number of chunks: {len(legal_chunks)}")
print(f"Average chunk size: {sum(len(c) for c in legal_chunks) // len(legal_chunks)} chars\n")

print("Sample chunks with metadata:\n")
for i, chunk in enumerate(legal_chunks[:4]):
    # Simple section detection
    section = "Unknown"
    if "DEFINITIONS" in chunk:
        section = "Definitions"
    elif "SCOPE" in chunk:
        section = "Scope of Services"
    elif "COMPENSATION" in chunk:
        section = "Compensation"
    elif "INTELLECTUAL PROPERTY" in chunk:
        section = "Intellectual Property"
    elif "CONFIDENTIALITY" in chunk:
        section = "Confidentiality"
    elif "TERMINATION" in chunk:
        section = "Termination"
    
    print(f"Chunk {i+1} | Section: {section} | Size: {len(chunk)} chars")
    print(f"Content: {chunk[:200]}...")
    print("-" * 60)

## ❌ Failure Case

Poor chunking strategies and their consequences.

In [ ]:
# Demonstrating chunking failures

problematic_text = """
Important Safety Warning: Do not mix cleaning products. Mixing bleach and ammonia creates toxic chloramine gas that can cause serious injury or death.

Always read labels carefully and ensure proper ventilation when using any cleaning chemicals.
"""

print("=== FAILURE 1: CHUNKS TOO SMALL ===")
tiny_splitter = CharacterTextSplitter(chunk_size=50, chunk_overlap=0)
tiny_chunks = tiny_splitter.split_text(problematic_text)
print(f"Chunks created: {len(tiny_chunks)}")
for i, chunk in enumerate(tiny_chunks):
    print(f"Chunk {i+1}: '{chunk}'")
print("\n⚠️ Problem: Critical safety warning split mid-sentence!\n")

print("=== FAILURE 2: NO OVERLAP ===")
no_overlap_text = """
The project timeline spans Q1 through Q4. Phase 1 begins in January and concludes in March. Phase 2 starts in April and runs through June.
"""
no_overlap_splitter = CharacterTextSplitter(chunk_size=80, chunk_overlap=0)
no_overlap_chunks = no_overlap_splitter.split_text(no_overlap_text)
print("Chunks:")
for i, chunk in enumerate(no_overlap_chunks):
    print(f"  {i+1}. {chunk}")
print("\n⚠️ Problem: Timeline continuity lost between chunks!\n")

print("=== FAILURE 3: WRONG SEPARATORS ===")
code_example = """
def calculate_total(items):
    total = 0
    for item in items:
        total += item.price
    return total
"""
bad_code_splitter = CharacterTextSplitter(
    separator=". ",  # Wrong separator for code!
    chunk_size=100,
    chunk_overlap=0
)
bad_code_chunks = bad_code_splitter.split_text(code_example)
print("Code chunks (BROKEN):")
for i, chunk in enumerate(bad_code_chunks):
    print(f"Chunk {i+1}: {repr(chunk)}")
print("\n⚠️ Problem: Code split at wrong boundaries - syntax broken!\n")

print("=== SOLUTION: PROPER CHUNKING ===")
good_code_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=20,
    separators=["\n\n", "\n", " ", ""],  # Respect line boundaries
)
good_code_chunks = good_code_splitter.split_text(code_example)
print("Properly chunked code:")
for i, chunk in enumerate(good_code_chunks):
    print(f"\nChunk {i+1}:")
    print(chunk)

## 📊 Benchmark Comparison

| Chunking Strategy | Retrieval Accuracy | Context Preservation | Processing Speed | Best For |
|-------------------|-------------------|---------------------|------------------|----------|
| **Fixed Character** | 65% | Poor | Fastest | Simple docs |
| **Recursive Char** | 78% | Good | Fast | General purpose |
| **Token-based** | 75% | Good | Fast | LLM-optimized |
| **Semantic** | 88% | Excellent | Slow | Complex docs |
| **Hierarchical** | 85% | Excellent | Medium | Structured docs |

### Chunk Size Impact:
| Chunk Size | Precision | Recall | Context | Use Case |
|------------|-----------|--------|---------|----------|
| 100 tokens | High | Low | Limited | Fact lookup |
| 300 tokens | Medium | Medium | Good | Q&A systems |
| 500 tokens | Low | High | Excellent | Summarization |

### Key Insights:
- Smaller chunks = higher precision, lower recall
- Larger chunks = better context, more noise
- 10-20% overlap is the sweet spot for most use cases
- Always match separators to content type

## 🎮 Interactive Playground

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║              DOCUMENT CHUNKING EXPERIMENT LAB                      ║
# ╚══════════════════════════════════════════════════════════════════════╝

print("Document Chunking Playground\n")

# Get user document
print("Enter your document text (press Enter twice when done):")
lines = []
while True:
    line = input()
    if line == "":
        break
    lines.append(line)
user_document = "\n".join(lines)

if not user_document.strip():
    print("Using sample document...")
    user_document = sample_document

print(f"\nDocument size: {count_tokens(user_document)} tokens\n")

# Choose strategy
print("Choose chunking strategy:")
print("1. Fixed Character")
print("2. Recursive Character")
print("3. Token-based")
strategy = input("Enter choice (1-3): ")

# Get parameters
chunk_size = int(input("\nChunk size (tokens/chars): ") or "300")
chunk_overlap = int(input("Overlap percentage (0-30): ") or "10")
overlap_size = int(chunk_size * chunk_overlap / 100)

# Create splitter
if strategy == "1":
    splitter = CharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap_size
    )
    strategy_name = "Fixed Character"
elif strategy == "3":
    splitter = TokenTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap_size
    )
    strategy_name = "Token-based"
else:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap_size
    )
    strategy_name = "Recursive Character"

# Chunk the document
chunks = splitter.split_text(user_document)

print(f"\n{'='*60}")
print(f"RESULTS: {strategy_name}")
print(f"{'='*60}")
print(f"Total chunks: {len(chunks)}")
print(f"Chunk size: {chunk_size}")
print(f"Overlap: {overlap_size} ({chunk_overlap}%)\n")

for i, chunk in enumerate(chunks):
    print(f"\n--- Chunk {i+1} ---")
    print(chunk[:300] + "..." if len(chunk) > 300 else chunk)

## 💡 Tips & Tricks

### Content-Type Specific Guidelines:

**Code:**
- Use line breaks (`\n`) as primary separator
- Keep function/class definitions intact
- Include docstrings with functions

**Legal/Medical:**
- Use paragraph breaks (`\n\n`)
- Higher overlap (15-25%) for clause continuity
- Include section headers in chunks

**Conversational:**
- Keep utterances together
- Maintain speaker attribution
- Use turn boundaries as separators

**Technical Documentation:**
- Preserve code blocks intact
- Keep examples with explanations
- Use heading hierarchy

### Advanced Techniques:
1. **Metadata Preservation**: Add source, page number, section to each chunk
2. **Parent Document Retrieval**: Retrieve chunks but use parent for context
3. **Hierarchical Chunking**: Multi-level chunks (section → paragraph → sentence)
4. **Semantic Chunking**: Use embeddings to find natural boundaries
5. **Agentic Chunking**: LLM decides optimal chunk boundaries

## 📚 References

### Research:
- [Late Chunking: Contextual Chunk Embeddings (Jina AI, 2024)](https://arxiv.org/abs/2409.04701)
- [Dense Passage Retrieval (Karpukhin et al., 2020)](https://arxiv.org/abs/2004.04906)

### Documentation:
- [LangChain Text Splitters](https://python.langchain.com/docs/modules/data_connection/document_transformers/)
- [LlamaIndex Node Parsing](https://docs.llamaindex.ai/en/stable/module_guides/loading/node_parsers/)

### Related Techniques:
- Basic RAG (Technique 53)
- Context Injection (Technique 54)
- Semantic Search (Technique 56)
- Hybrid Retrieval (Technique 57)